# P2P SmolVLM QLoRA — Colab H100, Paper-Aligned

**Config (per Schulman et al, LoRA Without Regret, 2025):**
- All-modules LoRA (attention + MLP), rank 8, alpha 32
- LR 2e-4 (~15x FullFT)
- Batch 8, no grad accum (effective batch 8, paper says LoRA prefers small batches)
- bf16 compute on H100
- 3 seeds × 3 epochs with score-averaging ensemble

**Workflow:**
1. Run **Cell 1** ONCE per session
2. Run **Cell 2** with `SEED = 42` → finishes ~85 min → **DOWNLOAD CSV TO LAPTOP** before doing anything else
3. Edit `SEED = 7`, run Cell 2 again, download CSV
4. Edit `SEED = 17`, run Cell 2 again, download CSV
5. Run **Cell 3** for ensemble → download ensemble CSV → submit all 4 to LB

**HARD RULE: After each seed completes, IMMEDIATELY click the file in left sidebar to download CSV. Drive saves are backup; local is primary truth.**

## Cell 1 — Setup (run ONCE per session)

In [6]:
!pip install nbstripout

In [2]:
# ─── Mount Drive ───
from google.colab import drive
drive.mount('/content/drive')

# ─── Drive paths ───
import os, shutil
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive')
ZIP_PATH = DRIVE_ROOT / 'Pixels_to_Predictions.zip'
OUT_DIR = DRIVE_ROOT / 'p2p_h100'
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / 'submissions').mkdir(exist_ok=True)
(OUT_DIR / 'scores').mkdir(exist_ok=True)
(OUT_DIR / 'adapters').mkdir(exist_ok=True)
(OUT_DIR / 'ckpts').mkdir(exist_ok=True)
print(f'Drive output dir: {OUT_DIR}')

# ─── Unzip data ───
DATA_DIR = Path('/content/data')
if not (DATA_DIR/'train.csv').exists():
    !cp '{ZIP_PATH}' /content/
    !unzip -q /content/Pixels_to_Predictions.zip -d /content/data/
print('data files:', sorted(os.listdir(DATA_DIR)))

# ─── GPU check ───
!nvidia-smi --query-gpu=name,memory.total --format=csv | head -3

# ─── Install ───
!pip install -q -U transformers==4.57.6 peft==0.18.1 bitsandbytes==0.48.1 accelerate==1.10.1

# ─── Imports ───
import json, gc, random, math, time
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ─── Locate images ───
_sample = pd.read_csv(DATA_DIR/'train.csv').iloc[0]['image_path']
IMG_ROOT = None
for cand in [DATA_DIR, DATA_DIR/'images', DATA_DIR.parent]:
    if (cand/_sample).exists():
        IMG_ROOT = cand; break
assert IMG_ROOT, f'cannot find image: {_sample}'
print(f'IMG_ROOT: {IMG_ROOT}')

train_df = pd.read_csv(DATA_DIR/'train.csv')
val_df   = pd.read_csv(DATA_DIR/'val.csv')
test_df  = pd.read_csv(DATA_DIR/'test.csv')
print(f'rows: train={len(train_df)} val={len(val_df)} test={len(test_df)}')

# ─── Load base model (4-bit, bf16 compute on H100) ───
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

processor = AutoProcessor.from_pretrained(MODEL_ID)
tok = processor.tokenizer
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,  # H100-native
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, quantization_config=bnb_cfg, torch_dtype=torch.bfloat16,
    device_map='auto', low_cpu_mem_usage=True,
)
print('✓ base model loaded')

# ─── Prompt + image helpers ───
LETTERS = ['A','B','C','D','E']
MAX_HINT, MAX_LEC = 600, 800
IMG_SIZE = 384

def build_user_text(row):
    parts = []
    h = row.get('hint')
    if isinstance(h, str) and h.strip():
        parts.append(f'Hint: {h.strip()[:MAX_HINT]}')
    l = row.get('lecture')
    if isinstance(l, str) and l.strip():
        parts.append(f'Context: {l.strip()[:MAX_LEC]}')
    choices = json.loads(row['choices'])
    letters = LETTERS[:len(choices)]
    cl = '\n'.join(f'  {ll}. {c}' for ll,c in zip(letters, choices))
    parts.append(f'Question: {row["question"]}\nChoices:\n{cl}\nReply with the single letter of the correct choice.')
    return '\n\n'.join(parts), choices, letters

def load_image(rel):
    img = Image.open(IMG_ROOT/rel).convert('RGB')
    img.thumbnail((IMG_SIZE,IMG_SIZE), Image.BICUBIC)
    bg = Image.new('RGB',(IMG_SIZE,IMG_SIZE),(255,255,255))
    bg.paste(img, ((IMG_SIZE-img.width)//2,(IMG_SIZE-img.height)//2))
    return bg

print('✓ Cell 1 done — Drive mounted, data ready, base model loaded')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive output dir: /content/drive/MyDrive/p2p_h100
data files: ['images', 'sample_submission.csv', 'test.csv', 'train.csv', 'val.csv']
/bin/bash: line 1: nvidia-smi: command not found
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.5 MB/s eta 0:00:00
IMG_ROOT: /content/data/images
rows: train=3109 val=1048 test=1008


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

✓ base model loaded
✓ Cell 1 done — Drive mounted, data ready, base model loaded


In [ ]:
!nvidia-smi

Sun May  3 17:49:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   31C    P0            115W /  700W |    1117MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!nvidia-smi

Sun May  3 17:49:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   31C    P0            114W /  700W |    1117MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Cell 2 — Train one seed + inference (run 3 times: 42, 7, 17)

**EDIT `SEED` for each run, then run all of Cell 2.**

**After each run finishes, IMMEDIATELY:**
1. Click left sidebar files icon
2. Find `submission_seed{N}.csv` and `test_scores_seed{N}.npy`
3. Right-click → Download to your laptop
4. Then upload CSV to Kaggle for LB submission

In [ ]:
# ════════════════════════════════════════════════
SEED = 17        # ← EDIT: 42 → 7 → 17
EPOCHS = 3
# ════════════════════════════════════════════════

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ─── Per-seed Drive paths ───
SEED_CKPT = OUT_DIR / 'ckpts' / f'seed{SEED}'
SEED_CKPT.mkdir(exist_ok=True)
ADAPTER_DIR = OUT_DIR / 'adapters' / f'seed{SEED}'
ADAPTER_DIR.mkdir(exist_ok=True)
print(f'==== SEED {SEED} ====')
print(f'  ckpts → {SEED_CKPT}')
print(f'  final adapter → {ADAPTER_DIR}')

# ─── Detach prior peft adapter from base ───
if isinstance(base_model, PeftModel):
    base_model = base_model.unload()

# ─── Prepare for training ───
model_b = prepare_model_for_kbit_training(base_model)
# Enable grad checkpointing to reduce activation memory (saves ~40% peak memory)
if hasattr(model_b, 'gradient_checkpointing_enable'):
    model_b.gradient_checkpointing_enable()
model_b.config.use_cache = False
print('✓ grad checkpointing enabled (saves memory across seeds)')

# ─── Resume detection ───
def find_latest_ckpt():
    ckpts = [p for p in SEED_CKPT.glob('step*') if p.is_dir() and not p.name.endswith('.tmp')]
    return max(ckpts, key=lambda p: int(p.name.replace('step',''))) if ckpts else None

latest = find_latest_ckpt()
if latest is not None:
    print(f'⟳ Resuming seed{SEED} from {latest}')
    model = PeftModel.from_pretrained(model_b, str(latest), is_trainable=True)
else:
    print(f'▶ Fresh seed{SEED}')
    # ALL-MODULES LoRA, rank 8, alpha 32 — paper-aligned
    lora_cfg = LoraConfig(
        r=8, lora_alpha=32, lora_dropout=0.05, bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj','k_proj','v_proj','o_proj',
                        'gate_proj','up_proj','down_proj'],
    )
    model = get_peft_model(model_b, lora_cfg)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
assert trainable <= 5_000_000, f'OVER 5M: {trainable}'
model.print_trainable_parameters()

# ─── Hyperparameters (paper-aligned, memory-tuned) ───
BATCH_SIZE = 2       # Reduced from 8 to fit in memory across seeds
GRAD_ACCUM = 4       # Effective batch = 2 × 4 = 8 (matches paper)
LR = 2e-4            # ~15x FullFT LR per paper (short runs)
MAX_GRAD_NORM = 1.0
SAVE_EVERY_STEPS = 25

# ─── Dataset + collate ───
class TrainDataset(Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        user_text, _, letters = build_user_text(row)
        return {
            'image': load_image(row['image_path']),
            'user_text': user_text,
            'gold_letter': letters[int(row['answer'])],
        }

def collate_train(batch):
    prompts_only, full_texts, images_list = [], [], []
    for ex in batch:
        msgs_user = [{'role':'user','content':[{'type':'image'},{'type':'text','text':ex['user_text']}]}]
        msgs_full = msgs_user + [{'role':'assistant','content':[{'type':'text','text':ex['gold_letter']}]}]
        prompts_only.append(processor.apply_chat_template(msgs_user, add_generation_prompt=True))
        full_texts.append(processor.apply_chat_template(msgs_full, add_generation_prompt=False))
        images_list.append([ex['image']])
    enc = processor(text=full_texts, images=images_list, return_tensors='pt', padding=True)
    prompt_enc = processor(text=prompts_only, images=images_list, return_tensors='pt', padding=True)
    plens = prompt_enc['attention_mask'].sum(dim=1).tolist()
    labels = enc['input_ids'].clone()
    for i, plen in enumerate(plens):
        labels[i, :plen] = -100
    labels[enc['attention_mask']==0] = -100
    out = {k: v for k,v in enc.items()}
    out['labels'] = labels
    return out

train_ds = TrainDataset(train_df)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_train, num_workers=2, pin_memory=True)

# ─── Optimizer + sched ───
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
optim = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
sched = CosineAnnealingLR(optim, T_max=max(1, total_steps))

# ─── Atomic checkpoint helpers (write to Drive directly) ───
def save_ckpt(step, epoch, batch_idx):
    tag = f'step{step:06d}'
    final = SEED_CKPT / tag
    tmp = SEED_CKPT / f'{tag}.tmp'
    if tmp.exists(): shutil.rmtree(tmp)
    tmp.mkdir()
    model.save_pretrained(tmp)
    torch.save({
        'optim': optim.state_dict(), 'sched': sched.state_dict(),
        'step': step, 'epoch': epoch, 'batch_idx': batch_idx,
        'rng_t': torch.get_rng_state(),
        'rng_c': torch.cuda.get_rng_state_all(),
        'rng_n': np.random.get_state(),
        'rng_p': random.getstate(),
    }, tmp/'state.pt')
    if final.exists(): shutil.rmtree(final)
    tmp.rename(final)
    # rolling: keep last 2
    ckpts = sorted([p for p in SEED_CKPT.glob('step*') if p.is_dir() and not p.name.endswith('.tmp')],
                   key=lambda p: int(p.name.replace('step','')))
    for old in ckpts[:-2]: shutil.rmtree(old)

# ─── Resume offsets ───
start_epoch, start_batch, step = 0, 0, 0
if latest is not None:
    ts = torch.load(latest/'state.pt', map_location='cpu', weights_only=False)
    optim.load_state_dict(ts['optim']); sched.load_state_dict(ts['sched'])
    torch.set_rng_state(ts['rng_t']); torch.cuda.set_rng_state_all(ts['rng_c'])
    np.random.set_state(ts['rng_n']); random.setstate(ts['rng_p'])
    step = ts['step']; start_epoch = ts['epoch']; start_batch = ts['batch_idx']+1
    print(f'  resume offset: epoch={start_epoch} batch={start_batch} step={step}')

# ─── Train loop ───
model.train()
loss_running, n_batches = 0.0, 0
t0 = time.time()
last_reminder = t0
for epoch in range(start_epoch, EPOCHS):
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f'seed{SEED} ep{epoch+1}/{EPOCHS}')
    optim.zero_grad()
    for i, batch in pbar:
        if epoch == start_epoch and i < start_batch: continue
        batch = {k:v.to(device) for k,v in batch.items()}
        out = model(**batch)
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        loss_running += loss.item() * GRAD_ACCUM
        n_batches += 1
        if (i+1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
            optim.step(); sched.step(); optim.zero_grad()
            step += 1
            pbar.set_postfix(loss=loss_running/max(1,n_batches), lr=sched.get_last_lr()[0], step=step)
            if step % SAVE_EVERY_STEPS == 0:
                save_ckpt(step, epoch, i)
        if time.time() - last_reminder > 1800:
            print(f'\n⏰ {int((time.time()-t0)/60)}min elapsed — checkpoints in Drive at {SEED_CKPT}')
            last_reminder = time.time()
    print(f'  epoch {epoch+1} mean_loss={loss_running/max(1,n_batches):.4f}')
    ep_dir = ADAPTER_DIR / f'epoch{epoch+1}'
    if ep_dir.exists(): shutil.rmtree(ep_dir)
    model.save_pretrained(ep_dir)
    print(f'  ✓ saved {ep_dir}')
    loss_running, n_batches, start_batch = 0.0, 0, 0

# Final adapter
FINAL = ADAPTER_DIR / 'final'
if FINAL.exists(): shutil.rmtree(FINAL)
model.save_pretrained(FINAL)
print(f'✓ FINAL adapter saved: {FINAL}')
model.eval()
torch.cuda.empty_cache(); gc.collect()

# ─── Inference: log-likelihood scoring ───
@torch.inference_mode()
def score_row(row):
    user_text, choices, letters = build_user_text(row)
    img = load_image(row['image_path'])
    msgs = [{'role':'user','content':[{'type':'image'},{'type':'text','text':user_text}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    enc = processor(text=[prompt], images=[[img]], return_tensors='pt').to(device)
    out = model(**enc)
    log_probs = F.log_softmax(out.logits[0,-1,:].float(), dim=-1)
    scores = []
    for L in letters:
        best = -1e9
        for c in [f' {L}', L]:
            ids = tok.encode(c, add_special_tokens=False)
            if len(ids) == 1:
                best = max(best, log_probs[ids[0]].item())
        scores.append(best)
    return int(np.argmax(scores)), scores

# Quick val sanity
n_ok = 0
for _, r in tqdm(val_df.head(100).iterrows(), total=100, desc=f'seed{SEED} val sanity'):
    p,_ = score_row(r); n_ok += int(p == int(r['answer']))
print(f'seed{SEED} val sanity 100: {n_ok/100:.3f}')

# Full val
val_preds = []
for _, r in tqdm(val_df.iterrows(), total=len(val_df), desc=f'seed{SEED} val'):
    p,_ = score_row(r); val_preds.append(p)
val_acc = (np.array(val_preds)==val_df['answer'].values).mean()
print(f'seed{SEED} VAL ACC: {val_acc:.4f}')

# Test
test_preds, test_scores = [], []
for _, r in tqdm(test_df.iterrows(), total=len(test_df), desc=f'seed{SEED} test'):
    p, s = score_row(r); test_preds.append(p)
    test_scores.append(s + [-1e9]*(5-len(s)))

# ─── Save outputs in 3 places: /content (download from sidebar), Drive (durable), and Drive submissions/scores dirs ───
sub = pd.DataFrame({'id': test_df['id'], 'answer': test_preds})
scores_arr = np.array(test_scores)

# /content for fast download from sidebar
sub.to_csv(f'/content/submission_seed{SEED}.csv', index=False)
np.save(f'/content/test_scores_seed{SEED}.npy', scores_arr)
# Drive — main copies
sub.to_csv(OUT_DIR/'submissions'/f'submission_seed{SEED}.csv', index=False)
np.save(OUT_DIR/'scores'/f'test_scores_seed{SEED}.npy', scores_arr)
# Save val acc for record
with open(OUT_DIR/f'val_acc_seed{SEED}.txt', 'w') as f:
    f.write(f'seed{SEED} val_acc={val_acc:.4f}\n')

print('\n' + '='*60)
print(f'✅ SEED {SEED} COMPLETE')
print(f'   val_acc: {val_acc:.4f}')
print(f'   answer dist: {sub["answer"].value_counts().to_dict()}')
print('='*60)
print('\n🚨 IMMEDIATE ACTIONS:')
print(f'  1. Left sidebar → Files → download:')
print(f'       /content/submission_seed{SEED}.csv')
print(f'       /content/test_scores_seed{SEED}.npy')
print(f'  2. Submit submission_seed{SEED}.csv to Kaggle LB')
print(f'  3. Verify Drive copy exists at: {OUT_DIR}/submissions/')
print(f'  4. Run cleanup cell, then edit SEED to next value (7 or 17), re-run Cell 2')

==== SEED 17 ====
  ckpts → /content/drive/MyDrive/p2p_h100/ckpts/seed17
  final adapter → /content/drive/MyDrive/p2p_h100/adapters/seed17
✓ grad checkpointing enabled (saves memory across seeds)
▶ Fresh seed17
trainable: 4,784,128 / 306,614,464 (1.56%)
trainable params: 4,784,128 || all params: 512,266,432 || trainable%: 0.9339


seed17 ep1/3:   0%|          | 0/1555 [00:00<?, ?it/s]

  epoch 1 mean_loss=0.2728
  ✓ saved /content/drive/MyDrive/p2p_h100/adapters/seed17/epoch1


seed17 ep2/3:   0%|          | 0/1555 [00:00<?, ?it/s]


⏰ 30min elapsed — checkpoints in Drive at /content/drive/MyDrive/p2p_h100/ckpts/seed17
  epoch 2 mean_loss=0.1253
  ✓ saved /content/drive/MyDrive/p2p_h100/adapters/seed17/epoch2


...

seed17 val sanity 100: 0.620


seed17 val:   0%|          | 0/1048 [00:00<?, ?it/s]

seed17 VAL ACC: 0.7853


seed17 test:   0%|          | 0/1008 [00:00<?, ?it/s]


✅ SEED 17 COMPLETE
   val_acc: 0.7853
   answer dist: {0: 356, 1: 344, 2: 228, 3: 75, 4: 5}

🚨 IMMEDIATE ACTIONS:
  1. Left sidebar → Files → download:
       /content/submission_seed17.csv
       /content/test_scores_seed17.npy
  2. Submit submission_seed17.csv to Kaggle LB
  3. Verify Drive copy exists at: /content/drive/MyDrive/p2p_h100/submissions/
  4. Run cleanup cell, then edit SEED to next value (7 or 17), re-run Cell 2


In [ ]:
# Compare epoch 2 vs epoch 3 adapters, save test scores from the better one
from peft import PeftModel
import gc

def evaluate_adapter(adapter_path, label):
    global model
    # Drop current model
    if 'model' in dir():
        try:
            base = model.unload() if isinstance(model, PeftModel) else base_model
        except:
            base = base_model
        del model
        gc.collect(); torch.cuda.empty_cache()
    else:
        base = base_model

    model = PeftModel.from_pretrained(base, str(adapter_path), is_trainable=False)
    model.eval()

    val_preds = []
    for _, r in tqdm(val_df.iterrows(), total=len(val_df), desc=f'{label} val'):
        p, _ = score_row(r); val_preds.append(p)
    acc = (np.array(val_preds) == val_df['answer'].values).mean()
    print(f'{label} val acc: {acc:.4f}')
    return acc

ep2_acc = evaluate_adapter(ADAPTER_DIR / 'epoch2', 'epoch2')
ep3_acc = evaluate_adapter(ADAPTER_DIR / 'epoch3', 'epoch3')

print(f'\nEpoch 2: {ep2_acc:.4f}')
print(f'Epoch 3: {ep3_acc:.4f}')
print(f'Winner: epoch{"2" if ep2_acc >= ep3_acc else "3"}')

# Re-run test inference with the winning adapter
best_path = ADAPTER_DIR / ('epoch2' if ep2_acc >= ep3_acc else 'epoch3')
if isinstance(model, PeftModel):
    model = model.unload()
gc.collect(); torch.cuda.empty_cache()
model = PeftModel.from_pretrained(base_model, str(best_path), is_trainable=False)
model.eval()

test_preds, test_scores = [], []
for _, r in tqdm(test_df.iterrows(), total=len(test_df), desc=f'seed{SEED} test (best)'):
    p, s = score_row(r); test_preds.append(p)
    test_scores.append(s + [-1e9]*(5-len(s)))

# Overwrite the saved files with the winner's predictions
sub = pd.DataFrame({'id': test_df['id'], 'answer': test_preds})
scores_arr = np.array(test_scores)
sub.to_csv(f'/content/submission_seed{SEED}.csv', index=False)
np.save(f'/content/test_scores_seed{SEED}.npy', scores_arr)
sub.to_csv(OUT_DIR/'submissions'/f'submission_seed{SEED}.csv', index=False)
np.save(OUT_DIR/'scores'/f'test_scores_seed{SEED}.npy', scores_arr)
print(f'\n✓ Saved predictions from {best_path.name} (val_acc={max(ep2_acc, ep3_acc):.4f})')

In [ ]:
# ─── CLEANUP CELL — run between seeds (no restart needed) ───
import gc, torch
from peft import PeftModel

# Unload the LoRA adapter back to base model
if 'model' in dir() and isinstance(model, PeftModel):
    base_model = model.unload()
    del model

# Drop training state
for name in ['optim', 'sched', 'train_loader', 'train_ds', 'model_b',
             'val_preds', 'test_preds', 'test_scores', 'scores_arr',
             'sub', 'enc', 'prompt_enc', 'batch', 'out', 'loss']:
    if name in dir():
        exec(f'del {name}')

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
gc.collect()
torch.cuda.empty_cache()

print('✓ Cleanup done. Memory state:')
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv
print('\nNow edit SEED in Cell 2 to the next value and re-run Cell 2.')


✓ Cleanup done. Memory state:
memory.used [MiB], memory.free [MiB]
2039 MiB, 79041 MiB

Now edit SEED in Cell 2 to the next value and re-run Cell 2.


## Cell 3 — Ensemble (run AFTER all 3 seeds done)

Combines per-seed score arrays via mean → final submission. Run this once at the end.

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

SEEDS = [42, 7, 17]

# Load each seed's scores from Drive
score_arrays = []
for s in SEEDS:
    p = OUT_DIR/'scores'/f'test_scores_seed{s}.npy'
    assert p.exists(), f'Missing: {p}'
    arr = np.load(p)
    score_arrays.append(arr)
    print(f'  seed {s}: {p} shape={arr.shape}')

stacked = np.stack(score_arrays, axis=0)  # (n_seeds, n_test, 5)
mean_scores = stacked.mean(axis=0)

# Mask invalid choices
for i, n in enumerate(test_df['num_choices'].values):
    mean_scores[i, n:] = -1e9

ens_preds = mean_scores.argmax(axis=1)
ens_sub = pd.DataFrame({'id': test_df['id'], 'answer': ens_preds})

# Save in both /content and Drive
ens_sub.to_csv('/content/submission_ensemble.csv', index=False)
ens_sub.to_csv(OUT_DIR/'submissions'/'submission_ensemble.csv', index=False)

print('\n✅ ENSEMBLE COMPLETE')
print('answer distribution:', ens_sub['answer'].value_counts().to_dict())

# Diagnostic: per-seed agreement
per_seed_preds = stacked.argmax(axis=2)
for i in range(len(SEEDS)):
    for j in range(i+1, len(SEEDS)):
        agree = (per_seed_preds[i] == per_seed_preds[j]).mean()
        print(f'  seeds {SEEDS[i]} vs {SEEDS[j]} agreement: {agree:.3f}')

# Compare ensemble vs each individual seed prediction
for i, s in enumerate(SEEDS):
    ind_preds = stacked[i].argmax(axis=1)
    diff = (ind_preds != ens_preds).sum()
    print(f'  ensemble vs seed{s}: {diff} different predictions')

print(f'\n🚨 DOWNLOAD /content/submission_ensemble.csv AND SUBMIT TO KAGGLE')

  seed 42: /content/drive/MyDrive/p2p_h100/scores/test_scores_seed42.npy shape=(1008, 5)
  seed 7: /content/drive/MyDrive/p2p_h100/scores/test_scores_seed7.npy shape=(1008, 5)
  seed 17: /content/drive/MyDrive/p2p_h100/scores/test_scores_seed17.npy shape=(1008, 5)

✅ ENSEMBLE COMPLETE
answer distribution: {0: 344, 1: 343, 2: 233, 3: 82, 4: 6}
  seeds 42 vs 7 agreement: 0.904
  seeds 42 vs 17 agreement: 0.875
  seeds 7 vs 17 agreement: 0.866
  ensemble vs seed42: 53 different predictions
  ensemble vs seed7: 74 different predictions
  ensemble vs seed17: 82 different predictions

🚨 DOWNLOAD /content/submission_ensemble.csv AND SUBMIT TO KAGGLE


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np, pandas as pd, json

ROOT = Path('/content/drive/MyDrive/p2p_h100')

# 1. Per-seed val accs
print("=== Per-seed val accuracies ===")
for s in [42, 7, 17]:
    f = ROOT / f'val_acc_seed{s}.txt'
    if f.exists(): print(f'  seed {s}: {f.read_text().strip()}')
    else: print(f'  seed {s}: NOT FOUND at {f}')

# 2. Which epoch was selected (look at adapter dirs that exist)
print("\n=== Adapter checkpoints saved per seed ===")
for s in [42, 7, 17]:
    adir = ROOT / 'adapters' / f'seed{s}'
    if adir.exists():
        eps = sorted([p.name for p in adir.iterdir() if p.is_dir()])
        print(f'  seed {s}: {eps}')

# 3. Param breakdown from one adapter (vision vs text, MLP vs attention)
from safetensors import safe_open
adapter_path = ROOT / 'adapters' / 'seed42' / 'epoch3' / 'adapter_model.safetensors'
if adapter_path.exists():
    print("\n=== Adapter parameter breakdown (seed42/epoch3) ===")
    vision_params = text_attn = text_mlp = 0
    with safe_open(adapter_path, framework='pt') as f:
        for k in f.keys():
            t = f.get_tensor(k)
            n = t.numel()
            if 'vision' in k.lower() or 'siglip' in k.lower():
                vision_params += n
            elif any(x in k for x in ['gate_proj','up_proj','down_proj']):
                text_mlp += n
            elif any(x in k for x in ['q_proj','k_proj','v_proj','o_proj']):
                text_attn += n
    total = vision_params + text_attn + text_mlp
    print(f'  Vision encoder LoRA: {vision_params:,}')
    print(f'  Text attention LoRA: {text_attn:,}')
    print(f'  Text MLP LoRA:       {text_mlp:,}')
    print(f'  Total:               {total:,}')
    print(f'  MLP fraction:        {text_mlp/total*100:.1f}%')
    print(f'  Attn fraction:       {(text_attn+vision_params)/total*100:.1f}%')

# 4. Train CSV stats (re-derive subject/grade/coverage)
print("\n=== Train CSV stats ===")
DATA = Path('/content/data')  # if not mounted, point to your CSVs
for split in ['train', 'val', 'test']:
    p = DATA / f'{split}.csv'
    if not p.exists(): continue
    df = pd.read_csv(p)
    print(f'\n  --- {split} (n={len(df)}) ---')
    if 'hint' in df.columns:
        hint_cov = df['hint'].notna().sum() / len(df) * 100
        print(f'    hint coverage:    {hint_cov:.1f}%')
    if 'lecture' in df.columns:
        lec_cov = df['lecture'].notna().sum() / len(df) * 100
        print(f'    lecture coverage: {lec_cov:.1f}%')
    if 'subject' in df.columns:
        print(f'    subject dist:     {df["subject"].value_counts(normalize=True).mul(100).round(1).to_dict()}')
    if split == 'train' and 'answer' in df.columns:
        print(f'    answer position dist: {df["answer"].value_counts(normalize=True).sort_index().mul(100).round(1).to_dict()}')

# 5. Recompute ensemble agreement triple-count from npy
print("\n=== Triple agreement breakdown (831/173/4 check) ===")
SEEDS = [42, 7, 17]
arrs = []
for s in SEEDS:
    p = ROOT / 'scores' / f'test_scores_seed{s}.npy'
    if p.exists(): arrs.append(np.load(p))
if len(arrs) == 3:
    preds = np.stack([a.argmax(1) for a in arrs])  # (3, 1008)
    all_agree = (preds[0]==preds[1]) & (preds[1]==preds[2])
    all_diff  = (preds[0]!=preds[1]) & (preds[0]!=preds[2]) & (preds[1]!=preds[2])
    two_agree = ~all_agree & ~all_diff
    n = preds.shape[1]
    print(f'  all agree:  {all_agree.sum()} ({all_agree.sum()/n*100:.1f}%)')
    print(f'  two agree:  {two_agree.sum()} ({two_agree.sum()/n*100:.1f}%)')
    print(f'  all differ: {all_diff.sum()} ({all_diff.sum()/n*100:.1f}%)')

# 6. Disagreement by num_choices (the 6.9/15.7/7.2/35.1 numbers)
test_df = pd.read_csv(DATA/'test.csv') if (DATA/'test.csv').exists() else None
if test_df is not None and 'num_choices' in test_df.columns and len(arrs)==3:
    print("\n=== Pairwise disagreement by num_choices ===")
    for nc in sorted(test_df['num_choices'].unique()):
        mask = (test_df['num_choices']==nc).values
        if mask.sum() == 0: continue
        diffs = []
        for i in range(3):
            for j in range(i+1, 3):
                diffs.append((preds[i][mask] != preds[j][mask]).mean())
        print(f'  {nc}-choice (n={mask.sum()}): mean disagreement {np.mean(diffs)*100:.1f}%')

Mounted at /content/drive
=== Per-seed val accuracies ===
  seed 42: seed42 val_acc=0.8025
  seed 7: seed7 val_acc=0.7767
  seed 17: seed17 val_acc=0.7853

=== Adapter checkpoints saved per seed ===
  seed 42: ['epoch1', 'epoch2', 'epoch3', 'final']
  seed 7: ['epoch1', 'epoch2', 'epoch3', 'final']
  seed 17: ['epoch1', 'epoch2', 'epoch3', 'final']

=== Adapter parameter breakdown (seed42/epoch3) ===
  Vision encoder LoRA: 442,368
  Text attention LoRA: 1,638,400
  Text MLP LoRA:       2,703,360
  Total:               4,784,128
  MLP fraction:        56.5%
  Attn fraction:       43.5%

=== Train CSV stats ===

=== Triple agreement breakdown (831/173/4 check) ===
  all agree:  831 (82.4%)
  two agree:  173 (17.2%)
  all differ: 4 (0.4%)


In [ ]:
import gc, torch, numpy as np
from peft import PeftModel
from tqdm.auto import tqdm
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/p2p_h100')
SEEDS = [42, 7, 17]
results = {}

# Make sure base is clean (no adapter attached)
if 'model' in dir() and isinstance(model, PeftModel):
    base_model = model.unload()
    del model
gc.collect(); torch.cuda.empty_cache()

def eval_adapter(adapter_path, label):
    global base_model
    m = PeftModel.from_pretrained(base_model, str(adapter_path), is_trainable=False)
    m.eval()
    # score_row uses the global `model`, so swap it in temporarily
    global model
    model = m
    preds = []
    for _, r in tqdm(val_df.iterrows(), total=len(val_df), desc=label, leave=False):
        p, _ = score_row(r); preds.append(p)
    acc = (np.array(preds) == val_df['answer'].values).mean()
    # detach this adapter cleanly
    base_model = m.unload()
    del m, model
    gc.collect(); torch.cuda.empty_cache()
    return acc

for s in SEEDS:
    adir = ROOT / 'adapters' / f'seed{s}'
    ep2 = adir / 'epoch2'
    ep3 = adir / 'epoch3'
    if not (ep2.exists() and ep3.exists()):
        print(f'seed {s}: missing epoch dirs, skipping')
        continue
    a2 = eval_adapter(ep2, f'seed{s} ep2')
    a3 = eval_adapter(ep3, f'seed{s} ep3')
    winner = 'epoch3' if a3 >= a2 else 'epoch2'
    results[s] = (a2, a3, winner)
    print(f'seed {s}: ep2={a2:.4f}  ep3={a3:.4f}  →  winner: {winner}')

print('\n=== SUMMARY ===')
for s, (a2, a3, w) in results.items():
    print(f'  seed {s}: ep2={a2:.4f} | ep3={a3:.4f} | selected: {w}')
all_ep3 = all(w == 'epoch3' for _, _, w in results.values())
print(f'\nReport claim "epoch-3 selected for all three seeds": {"✓ CONFIRMED" if all_ep3 else "✗ NEEDS UPDATE"}')